In [14]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// --- Kernel Naive ---
__global__ void matMulNaive(const float * a, const float * b, float * c, int n) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n) {
        float sum = 0.0f;
        //#pragma unroll
        for (int k = 0; k < n; ++k) {
            sum += a[row * n + k] * b[k * n + col];
        }
        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    if (n <= 0) return 1;

    // CPU Memory Allocation
    size_t bytes = (size_t)n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // GPU Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy data from host to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Grid and block dimensions
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation...\\n");

    // CUDA Event for measuring execution time
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch of the kernel + timing
    cudaEventRecord(start);
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaEventRecord(stop);

    // Wait for the kernel to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\\n", milliseconds / 1000.0f);

    // Copy data from device to host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample_size = (n < 1000) ? n : 1000;
        for (int i = 0; i < sample_size; i++) {
            for (int j = 0; j < sample_size; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    // Deallocation: both CPU and GPU
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [15]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o cuda_matrixmult_naive
!nvprof ./cuda_matrixmult_naive 20000

==7834== NVPROF is profiling process 7834, command: ./cuda_matrixmult_naive 20000
Starting the computation...
Execution Time: 28.0479 seconds
==7834== Profiling application: ./cuda_matrixmult_naive 20000
==7834== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.49%  28.0475s         1  28.0475s  28.0475s  28.0475s  matMulNaive(float const *, float const *, float*, int)
                    2.33%  677.03ms         2  338.52ms  336.52ms  340.51ms  [CUDA memcpy HtoD]
                    1.18%  342.70ms         1  342.70ms  342.70ms  342.70ms  [CUDA memcpy DtoH]
      API calls:   95.88%  28.0475s         1  28.0475s  28.0475s  28.0475s  cudaEventSynchronize
                    3.49%  1.02065s         3  340.22ms  336.80ms  343.13ms  cudaMemcpy
                    0.60%  176.34ms         3  58.781ms  138.85us  176.06ms  cudaMalloc
                    0.02%  6.2245ms         3  2.0748ms  1.3706ms  2.8424ms  cudaFree
   